<a href="https://colab.research.google.com/github/lelabbady/TReND-CaMinA/blob/main/Project_5_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 5 — Comparing Stimulus Response Properties for a Population of Neurons

### Why compare responses across stimuli?

Visual cortex neurons are often described using tuning metrics measured with simple
stimuli like gratings, but in practice those same neurons also respond to richer,
naturalistic inputs. A key question is whether these response properties are
consistent across stimulus classes, or whether each stimulus reveals different
functional structure in the population.

In this project, you will compare activity patterns from the same set of neurons
across drifting gratings, static gratings, and natural scenes.
This lets us ask both descriptive and mechanistic questions about selectivity,
responsiveness, and similarity of neural representations.

<div style="border-radius: 3px; padding: 10px;">

<b>Goal.</b> Build intuition for how population response structure depends on stimulus type.
We will work through:
<ol>
<li>Tuning curves to <b>drifting gratings</b> (direction × temporal frequency)</li>
<li>Responses to <b>static gratings</b> and <b>natural scenes</b></li>
<li>How to define whether a cell is <b>responsive</b> (using multiple criteria)</li>
<li>Whether drifting-grating responders also respond to <b>natural images</b></li>
<li>Whether cells with similar tuning also show similar <b>natural-movie</b> responses</li>
</ol>

Some <b>helpful publications</b> to read throughout this project:
<ol>
<li>A large-scale standardized physiological survey reveals functional organization of the mouse visual cortex. de Vries et al., Nature 2019</li>
<li>The effect of inclusion criteria on the functional properties reported in mouse visual cortex. Mesa, Waters, and de Vries, eNeuro 2021</li>
</ol>
<p>All data-access and helper functions are defined in the notebook itself, so you can read,
edit, and build on them directly as you go.</p>

</div>

### For Collab

In [ ]:
# @title Run to initialize Allen Brain Observatory on Colab {display-mode: "form" }

# run only once per runtime/session, and only if running in colab
# the runtime will need to restxart after
%%capture
!apt install s3fs

!pip uninstall -y numpy pandas
!pip install git+https://github.com/AllenInstitute/AllenSDK@1bdca3ad884c3a5edea8236161424650603e6f29 "numpy == 1.26.4" "pandas == 2.3.0" "matplotlib > 3.8.0" "statsmodels >= 0.14.4"
import allensdk
print('allensdk imported successfully')

!mkdir -p /data/allen-brain-observatory/
!s3fs allen-brain-observatory /data/allen-brain-observatory/ -o public_bucket=1

import time
print("Runtime is now restarting...")
print("You can ignore the error message [Your session crashed for an unknown reason.]")
time.sleep(5)
exit()


### Standard imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
from allensdk.core.brain_observatory_cache import BrainObservatoryCache

%matplotlib inline

<div style="border-radius: 3px; padding: 10px;">
Connect to the Allen Brain Observatory cache.
</div>

In [ ]:
import platform, os, sys
platstring = platform.platform()

if ('amzn' in platstring) or ('google.colab' in sys.modules):
    # for AWS
    vc_cache_dir = '/data/allen-brain-observatory/visual-coding-2p'
else:
    # for local drive, different operating systems
    if ('Darwin' in platstring) or ('macOS' in platstring):
        # OS X
        data_root = "/Volumes/TReND2026/"
    elif 'Windows'  in platstring:
        # Windows (replace with the drive letter of USB drive)
        data_root = "E:/"
    else:
        # your own linux platform
        # EDIT location where you mounted hard drive
        data_root = "/media/$USERNAME/TReND2026/"

    # visual behavior cache directory
    vc_cache_dir = os.path.join(data_root, "visual_coding_2p")

boc = BrainObservatoryCache(manifest_file=os.path.join(vc_cache_dir, 'manifest.json'))

<div style="border-radius: 3px; padding: 10px;">

<b>Choosing a population.</b> We will start by looking at inhibitory SST neurons in primary
visual cortex (V1), labelled by the <code>Sst-IRES-Cre</code> driver line in area
<code>VISp</code>. Feel free to change these later to explore other cell types
(e.g. the Layer 2/3 excitatory neurons <code>Cux2-CreERT2</code>, or other inhibitory lines like <code>Pvalb-IRES-Cre</code>,
<code>Vip-IRES-Cre</code>).

</div>

In [ ]:
CRE_LINE = 'Sst-IRES-Cre'
AREA = 'VISp'
DEPTHS = None

<div style="border-radius: 3px; padding: 10px;">

<b>Important — how the stimuli are organized.</b> Each population is imaged in three
separate sessions, and different stimuli live in different sessions:
<ul>
<li><b>Session A</b>: drifting_gratings, natural_movie_one, natural_movie_three</li>
<li><b>Session B</b>: static_gratings, natural_scenes, natural_movie_one</li>
<li><b>Session C</b>: locally_sparse_noise, natural_movie_one, natural_movie_two</li>
</ul>
So drifting gratings and natural scenes are <i>never</i> in the same session. To compare
a cell across stimuli we match the same <code>cell_specimen_id</code> across sessions.
    

<p>Let's start by picking a drifting-gratings session for our population and look at its metadata.

</div>

In [ ]:
exps = boc.get_ophys_experiments(cre_lines=[CRE_LINE], targeted_structures=[AREA],
                                 stimuli=['drifting_gratings'], imaging_depths = DEPTHS)
dg_session = exps[0]['id']
print("Drifting-gratings session:", dg_session)
boc.get_ophys_experiment_data(dg_session).get_metadata()

<div style="border-radius: 3px; padding: 10px;">
What does the metadata tell you about this population — the brain region, imaging depth (which cortical layer?), which cell-types were recorded?
</div>

## 1. Tuning curves to drifting gratings

<div style="border-radius: 3px; padding: 10px;">

Load every cell's ΔF/F for this session at once, plus the drifting-gratings stimulus
table. Look back at your Tuning Curver Tutorial to piece this together. <code>dff</code> has shape (n_cells, n_frames); <code>cell_ids</code> lines up with
its rows.

</div>

In [ ]:
def get_session_dff_and_stim(boc, session_id, stimulus):
    """ΔF/F for every cell in a session + the stimulus table.

    Returns
    -------
    timestamps : np.ndarray
    dff        : np.ndarray, shape (n_cells, n_frames)
    cell_ids   : np.ndarray, shape (n_cells,)  -- aligned with rows of `dff`
    stim_table : pandas.DataFrame
    """
    data_set = boc.get_ophys_experiment_data(session_id)
    timestamps, dff = data_set.get_dff_traces()
    cell_ids = np.array(data_set.get_cell_specimen_ids())
    stim_table = data_set.get_stimulus_table(stimulus)
    return timestamps, dff, cell_ids, stim_table

In [ ]:
# Test the helper on the drifting-gratings session and inspect the outputs.
timestamps, dff, cell_ids, dg_stim = get_session_dff_and_stim(boc, dg_session, 'drifting_gratings')
print('timestamps shape:', timestamps.shape)
print('dff shape:', dff.shape)
print('number of cells:', len(cell_ids))
print('stimulus table shape:', dg_stim.shape)


<div style="border-radius: 3px; padding: 10px;">

Now let's define <code>compute_trial_responses()</code> which gives the mean ΔF/F for every cell on every
stimulus presentation. Look back at your Tuning Curve Tutorial for help with this.

</div>

In [ ]:
def compute_trial_responses(dff, stim_table, statistic='mean', baseline_frames=0):
    """Per-presentation response for every cell.

    Parameters
    ----------
    dff : np.ndarray
        ``(n_cells, n_frames)`` or ``(n_frames,)`` for a single cell.
    stim_table : DataFrame
        Must have integer ``start`` / ``end`` frame columns.
    statistic : {'mean', 'max', 'sum'}
        How to summarise ΔF/F within each presentation window. Default 'mean'.
    baseline_frames : int
        If > 0, subtract the mean ΔF/F of the ``baseline_frames`` frames
        immediately before each presentation (a simple baseline correction).

    Returns
    -------
    responses : np.ndarray, shape (n_cells, n_trials)
        (For single-cell input it is still 2-D with one row; index ``[0]``.)
    """
    dff = np.atleast_2d(dff)
    starts = stim_table['start'].values.astype(int)
    ends = stim_table['end'].values.astype(int)
    func = {'mean': np.mean, 'max': np.max, 'sum': np.sum}[statistic]

    responses = np.zeros((dff.shape[0], len(stim_table)))
    for ix, (s, e) in enumerate(zip(starts, ends)):
        resp = func(dff[:, s:e], axis=1)
        if baseline_frames > 0:
            base = dff[:, max(0, s - baseline_frames):s].mean(axis=1)
            resp = resp - base
        responses[:, ix] = resp
    return responses


In [ ]:
# Can you test this function out with dff and pick an example cell from that set?
# Pick the cell with the strongest response as your example cell.
responses = compute_trial_responses(dff, dg_stim)          # (n_cells, n_trials)
strongest = cell_ids[np.argmax(responses.max(axis=1))]
print("Strongest-responding cell in this session:", strongest)

# We follow this specific cell through Sections 2 & 3 because it was recorded for
# drifting gratings, static gratings AND natural scenes (so every step below works).
example_cell = 588446511

<div style="border-radius: 3px; padding: 10px;">
Compute and plot this cell's 2-D tuning (direction × temporal frequency) with the provided helpers (these plots will look familiar).
</div>

In [ ]:
def get_dff_traces_and_stim_table(boc, cell_specimen_id, stimulus):
    """ΔF/F trace + stimulus table for ONE cell and ONE stimulus.

    Returns
    -------
    timestamps : np.ndarray
    dff_trace  : np.ndarray, shape (n_frames,)
    stim_table : pandas.DataFrame
    """
    exps = boc.get_ophys_experiments(cell_specimen_ids=[cell_specimen_id],
                                     stimuli=[stimulus])
    if len(exps) == 0:
        raise ValueError(
            f"No experiment found for cell {cell_specimen_id} with stimulus "
            f"'{stimulus}'. (Remember: drifting gratings and static gratings / "
            f"natural scenes live in different sessions.)")
    session_id = exps[0]['id']
    data_set = boc.get_ophys_experiment_data(session_id)
    timestamps, dff = data_set.get_dff_traces(cell_specimen_ids=[cell_specimen_id])
    dff_trace = dff[0, :]
    stim_table = data_set.get_stimulus_table(stimulus)
    return timestamps, dff_trace, stim_table

def drifting_gratings_tuning(boc, cell_specimen_id):
    """Direction x temporal-frequency tuning for one cell.

    Returns
    -------
    orivals : np.ndarray (directions, deg)
    tfvals  : np.ndarray (temporal frequencies, Hz)
    tuning  : np.ndarray, shape (n_ori, n_tf)  -- mean ΔF/F per condition
    blank_mean : float  -- mean ΔF/F on blank sweeps
    """
    _, dff_trace, stim_table = get_dff_traces_and_stim_table(
        boc, cell_specimen_id, 'drifting_gratings')
    resp = compute_trial_responses(dff_trace, stim_table)[0]

    ori = stim_table['orientation'].values
    tf = stim_table['temporal_frequency'].values
    orivals = np.unique(ori[np.isfinite(ori)])
    tfvals = np.unique(tf[np.isfinite(tf)])

    tuning = np.full((len(orivals), len(tfvals)), np.nan)
    # For each condition (a direction paired with a temporal frequency), average this
    # cell's response across the trials that showed it.
    for i, this_direction in enumerate(orivals):
        for j, this_tf in enumerate(tfvals):
            is_this_condition = (ori == this_direction) & (tf == this_tf)
            if is_this_condition.any():
                tuning[i, j] = resp[is_this_condition].mean()

    blank = np.isnan(ori)  # blank sweeps have NaN orientation
    blank_mean = resp[blank].mean() if blank.any() else np.nan
    return orivals, tfvals, tuning, blank_mean

def plot_tuning_heatmap(tuning, xvals, yvals, xlabel, ylabel, title,
                        zscore=True, save_path=None, ax=None):
    """Heatmap of a 2-D tuning array (e.g. direction x TF). z-scores by default
    so different cells are comparable."""
    import matplotlib.pyplot as plt

    arr = tuning.copy()
    if zscore:
        arr = (arr - np.nanmean(arr)) / np.nanstd(arr)
        cbar_label = "Z-score (std dev)"
        vmax = np.nanmax(np.abs(arr))
        vmin = -vmax
    else:
        cbar_label = "Mean DF/F"
        vmin = vmax = None

    if ax is None:
        _, ax = plt.subplots()
    im = ax.imshow(arr, cmap="RdBu_r", vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(xvals)))
    ax.set_yticks(range(len(yvals)))
    ax.set_yticklabels([f"{v:g}" for v in yvals])
    ax.set_xticklabels([f"{v:g}" for v in xvals])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    if save_path is not None:
        ax.figure.savefig(save_path)
    return ax

In [ ]:
# Step 1: compute the drifting-gratings tuning for the example cell.
# Step 2: plot the 2-D tuning heatmap with direction on the y-axis and temporal frequency on the x-axis.
orivals, tfvals, tuning, blank_mean = drifting_gratings_tuning(boc, example_cell)
plot_tuning_heatmap(tuning, tfvals, orivals, 'Temporal frequency (Hz)', 'Direction (deg)',
                    f'Drifting gratings tuning: cell {example_cell}', zscore=True)


<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 1a.</b> Collapse the 2-D tuning to a 1-D <i>direction</i> tuning curve (average across temporal frequency) and plot it with the blank-sweep reference line.
</div>

In [ ]:
def plot_tuning_curve(xvals, tuning, blank_mean=None, errors=None,
                      xlabel="Condition", ylabel="Mean DF/F", ax=None):
    """1-D tuning curve with optional error bars and a blank-sweep reference line."""
    import matplotlib.pyplot as plt

    if ax is None:
        _, ax = plt.subplots()
    if errors is None:
        ax.plot(xvals, tuning, 'o-')
    else:
        ax.errorbar(xvals, tuning, yerr=errors, fmt='o-')
    if blank_mean is not None and np.isfinite(blank_mean):
        ax.axhline(blank_mean, ls='--', color='k', label='blank')
        ax.legend()
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return ax

In [ ]:
# Step 1: average the 2-D tuning across temporal frequency to get a 1-D direction curve.
# Step 2: plot the direction tuning with the blank response as a reference line.
direction_tuning = np.nanmean(tuning, axis=1)
plot_tuning_curve(orivals, direction_tuning, blank_mean=blank_mean,
                  xlabel='Direction (deg)', ylabel='Mean DF/F')


<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 1b.</b> What is this cell's preferred direction and preferred temporal frequency?
</div>

In [ ]:
# Step 1: find the condition with the largest response in the 2-D tuning array.
# Step 2: decode that index back into the preferred direction and temporal frequency.
pref_idx = np.unravel_index(np.nanargmax(tuning), tuning.shape)
preferred_direction = orivals[pref_idx[0]]
preferred_tf = tfvals[pref_idx[1]]
print('Preferred direction:', preferred_direction)
print('Preferred temporal frequency:', preferred_tf)


<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 1c.</b> Plot tuning heatmaps for the six most strongly-responding cells in this session. What kinds of differences do you see between them?
</div>

In [ ]:
# Step 1: choose the strongest-responding cells you want to inspect.
# Step 2: build a 3-D array that stores direction x TF x cell tuning for the whole session.
# Step 3: fill one condition at a time by averaging across repeated trials.
# Step 4: identify the six cells with the largest responses.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


In [ ]:
# Step 1: loop over the cells you selected in the previous cell.
# Step 2: index the precomputed tuning array for each one.
# Step 3: plot each heatmap with the helper above.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


<div style="border-radius: 3px; padding: 10px;">
What about their tuning curves? Plot the average tuning curve for each cell. What information is maintained with these curves, what information is lost?
</div>

In [ ]:
# Step 1: average each selected cell's tuning across temporal frequency.
# Step 2: plot the resulting 1-D curve for each cell.
# Step 3: compare how much condition-specific detail is preserved versus the heatmap.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


## 2. Responses to static gratings and natural scenes

<div style="border-radius: 3px; padding: 10px;">

These stimuli live in <b>Session B</b>, not the drifting-gratings session you used above.
Conveniently, <code>get_dff_traces_and_stim_table()</code> looks a cell up by its
<code>cell_specimen_id</code> and automatically finds whichever session holds the stimulus
you ask for — so we can keep working with the <b>same example cell</b> from Section 1.

</div>

<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 2a.</b> Compute this cell's static-gratings tuning (orientation × spatial frequency) as a heatmap. How does this compare to the tuning for drifting grating? Are we measuring the same thing here?
</div>

In [ ]:
# Take a look at the information we get from static gratings for this one cell
# How does it compare?

_, dff_trace, stim_table = get_dff_traces_and_stim_table(
    boc, example_cell, 'static_gratings')
resp = compute_trial_responses(dff_trace, stim_table)[0]

In [ ]:
def static_gratings_tuning(boc, cell_specimen_id, collapse_phase=True):
    """Orientation x spatial-frequency tuning for one cell.

    Static gratings vary orientation, spatial_frequency and phase. By default we
    average over phase (``collapse_phase=True``) to get a 2-D ori x SF curve. Set
    ``collapse_phase=False`` to return a 3-D array including phase.

    Returns (collapse_phase=True)
    -----------------------------
    orivals, sfvals, tuning[n_ori, n_sf], blank_mean
    Returns (collapse_phase=False)
    ------------------------------
    orivals, sfvals, phasevals, tuning[n_ori, n_sf, n_phase], blank_mean
    """
    _, dff_trace, stim_table = get_dff_traces_and_stim_table(
        boc, cell_specimen_id, 'static_gratings')
    resp = compute_trial_responses(dff_trace, stim_table)[0]

    ori = stim_table['orientation'].values
    sf = stim_table['spatial_frequency'].values
    phase = stim_table['phase'].values
    orivals = np.unique(ori[np.isfinite(ori)])
    sfvals = np.unique(sf[np.isfinite(sf)])
    blank = np.isnan(ori)
    blank_mean = resp[blank].mean() if blank.any() else np.nan

    if collapse_phase:
        tuning = np.full((len(orivals), len(sfvals)), np.nan)
        # For each condition (an orientation paired with a spatial frequency), average
        # this cell's response across the trials that showed it.
        for i, this_orientation in enumerate(orivals):
            for j, this_sf in enumerate(sfvals):
                is_this_condition = (ori == this_orientation) & (sf == this_sf)
                if is_this_condition.any():
                    tuning[i, j] = resp[is_this_condition].mean()
        return orivals, sfvals, tuning, blank_mean

    phasevals = np.unique(phase[np.isfinite(phase)])
    tuning = np.full((len(orivals), len(sfvals), len(phasevals)), np.nan)
    # Same idea, but now keeping phase as a third dimension.
    for i, this_orientation in enumerate(orivals):
        for j, this_sf in enumerate(sfvals):
            for k, this_phase in enumerate(phasevals):
                is_this_condition = (ori == this_orientation) & (sf == this_sf) & (phase == this_phase)
                if is_this_condition.any():
                    tuning[i, j, k] = resp[is_this_condition].mean()
    return orivals, sfvals, phasevals, tuning, blank_mean

In [ ]:
# Step 1: compute the static-gratings tuning for the example cell.
# Step 2: collapse phase if needed and plot the orientation x spatial-frequency heatmap.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 2b.</b> Now compute this cell's response to each natural scene. Again - note what information is available. How might we compare these reponses to natural scenes to the ones to drifting and static gratings? Plot the response per image with the blank (mean-gray) response as a reference line, and report the preferred image.
</div>

In [ ]:
def natural_scenes_tuning(boc, cell_specimen_id):
    """Mean ΔF/F response to each natural-scene image for one cell.

    The natural-scenes stimulus table uses ``frame`` as the image id; frame
    ``-1`` is the blank (mean-gray) condition and is reported separately.

    Returns
    -------
    image_ids  : np.ndarray (sorted image ids, blank excluded)
    tuning     : np.ndarray, shape (n_images,)  -- mean ΔF/F per image
    blank_mean : float
    """
    _, dff_trace, stim_table = get_dff_traces_and_stim_table(
        boc, cell_specimen_id, 'natural_scenes')
    resp = compute_trial_responses(dff_trace, stim_table)[0]

    frames = stim_table['frame'].values
    image_ids = np.unique(frames[frames >= 0])  # -1 == blank
    tuning = np.array([resp[frames == img].mean() for img in image_ids])
    blank_mean = resp[frames == -1].mean() if (frames == -1).any() else np.nan
    return image_ids, tuning, blank_mean

In [ ]:
# Step 1: compute the response of the example cell to each natural-scene image.
# Step 2: unpack the image ids, tuning values, and blank-sweep reference.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


In [ ]:
# Step 1: plot the response for each image as a bar chart.
# Step 2: add the blank-sweep reference line and inspect the strongest/suppressed images.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


In [ ]:
# Step 1: use the surrounding markdown prompt to decide the analysis you want to run.
# Step 2: keep the result aligned with the same example cell or matched population.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


<div style="border-radius: 3px; padding: 10px;">
How does quantifying the response differ across these stimuli? Drifting gratings have long (2 s) presentations and 40 conditions; static gratings are brief with orientation/SF/phase; natural scenes have 118 distinct images shown briefly. Should 'responsive' be defined the same way for each?
</div>

## 3. How would you decide if a cell is responsive?

<div style="border-radius: 3px; padding: 10px;">

There is no single right answer — below are two potential criteria you miight use:
<ul>
<li><b>A — statistical</b> (<code>is_responsive_ttest</code>): the best condition drives
responses significantly above blank sweeps (t-test, corrected for multiple conditions).</li>
<li><b>B — amplitude</b> (<code>is_responsive_amplitude</code>): the peak response exceeds
blank by an absolute ΔF/F threshold.</li>
</ul>

</div>

<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 3a.</b> Use the helper functions before to apply criteria A and B to your example cell's drifting-gratings responses. Do they agree?
</div>

In [ ]:
def is_responsive_ttest(resp, condition_ids, blank_mask, alpha=0.05):
    """Criterion A (statistical): a cell is responsive if its *best* stimulus
    condition drives responses significantly greater than blank sweeps.

    One-sided Welch t-test per condition vs. blank, Bonferroni-corrected across
    conditions.

    Parameters
    ----------
    resp : np.ndarray, shape (n_trials,)  -- single-cell trial responses
    condition_ids : np.ndarray, shape (n_trials,)
        Condition label per trial (e.g. orientation, or image id). Blank trials
        may carry any value; they are excluded via ``blank_mask``.
    blank_mask : np.ndarray of bool, shape (n_trials,)
        True for blank-sweep trials.

    Returns
    -------
    (responsive: bool, best_corrected_p: float)
    """
    from scipy import stats

    blank = resp[blank_mask]
    labels = condition_ids[~blank_mask]
    rr = resp[~blank_mask]
    if len(blank) < 2:
        return False, np.nan

    pvals = []
    for c in np.unique(labels):
        trials = rr[labels == c]
        if len(trials) > 1:
            t, p = stats.ttest_ind(trials, blank, equal_var=False)
            p = p / 2 if t > 0 else 1 - (p / 2)   # one-sided (greater)
            pvals.append(p)
    if not pvals:
        return False, np.nan
    best = min(np.min(pvals) * len(pvals), 1.0)    # Bonferroni
    return bool(best < alpha), float(best)


def is_responsive_amplitude(tuning, blank_mean, min_dff=0.05):
    """Criterion B (amplitude): responsive if the peak condition response exceeds
    the blank-sweep response by an absolute ΔF/F threshold.

    Returns ``(responsive: bool, peak_minus_blank: float)``.
    """
    peak = np.nanmax(tuning)
    delta = peak - blank_mean
    return bool(delta >= min_dff), float(delta)

In [ ]:
# Step 1: calculate the drifting-gratings tuning for the example cell.
# Step 2: keep the tuning array and blank mean for the responsiveness tests.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


In [ ]:
# Step 1: test criterion A using the per-trial responses and condition labels.
# Step 2: test criterion B using the tuning array and blank mean.
# Step 3: compare the two labels for this cell.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 3b.</b> Now scale from a single cell to <i>every</i> cell in a session — in one shot.
Below we have defined <code>compute_tuning</code> to load a whole session and the tuning of all of its cells at once. Note that it is generalised to take the <b>stimulus</b> as an argument and to return each
cell's <code>cell_specimen_id</code>.


Use it to apply criteria A and B to every cell — how many are
'responsive' under each, and how often do the two agree?
</div>

In [ ]:
def compute_tuning(boc, session_id, stimulus='drifting_gratings'):
    """Every cell in a session at once (the fast Project-2 move), for ANY stimulus.

    Loads the session's DF/F a single time and vectorises across cells, instead of
    looping get_dff_traces_and_stim_table cell-by-cell. Returns each cell's response
    to each stimulus condition, with the blank (gray-screen) response subtracted.

    Returns
    -------
    cell_ids   : (n_cells,)              -- cell_specimen_ids, aligned to the cell axis
    responses  : (n_cells, n_trials)     -- per-presentation mean DF/F
    condition  : (n_trials,)             -- condition label for each trial
    blank      : (n_trials,) bool        -- True for blank / gray-screen trials
    condvals   : (n_conditions,)         -- the unique (non-blank) condition values
    tuning     : (n_conditions, n_cells) -- mean DF/F per condition, blank-subtracted
    blank_mean : (n_cells,)              -- each cell's mean blank response
    """
    _, dff, cell_ids, stim = get_session_dff_and_stim(boc, session_id, stimulus)
    responses = compute_trial_responses(dff, stim)            # (n_cells, n_trials)

    # The one thing that differs per stimulus: which column is the "condition", and
    # how the blank (gray-screen) trials are marked.
    if stimulus == 'natural_scenes':
        condition = stim['frame'].values         # each image is a condition
        blank = condition == -1                  # frame -1 == blank
    else:                                         # drifting / static gratings
        condition = stim['orientation'].values
        blank = np.isnan(condition)              # blank sweeps have no orientation

    blank_mean = responses[:, blank].mean(axis=1)            # each cell's mean blank response
    condvals = np.unique(condition[~blank])                  # the unique stimulus conditions

    # For each condition, average every cell's response across the trials that showed it,
    # then subtract that cell's blank response. Fill the table row by row.
    tuning = np.full((len(condvals), len(cell_ids)), np.nan)
    for k, this_condition in enumerate(condvals):
        is_this_condition = condition == this_condition
        tuning[k, :] = responses[:, is_this_condition].mean(axis=1) - blank_mean   # all cells at once
    return cell_ids, responses, condition, blank, condvals, tuning, blank_mean

In [ ]:
# Step 1: compute tuning for the full drifting-gratings session.
# Step 2: keep the cell ids, per-trial responses, condition labels, and blank mask.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


In [ ]:
# Step 1: apply criterion B to every cell in the session.
# Step 2: compare the amplitude-based labels to criterion A.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


In [ ]:
# Step 1: use the surrounding markdown prompt to decide the analysis you want to run.
# Step 2: keep the result aligned with the same example cell or matched population.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 3b (continued) — other stimuli.</b> Because <code>compute_tuning</code> takes the
stimulus as an argument, you can measure responsiveness for the other stimuli with the <i>same</i>
code — just grab a session that contains them (exactly as we grabbed <code>dg_session</code> in
Section 1) and pass the stimulus name. Try <code>static_gratings</code> and
<code>natural_scenes</code>. Are similar fractions of cells responsive?
</div>

In [ ]:
# compute_tuning takes the stimulus as an argument, so the SAME recipe works everywhere.
for stimulus in ['static_gratings', 'natural_scenes']:
    # grab a session that has this stimulus (same pattern as dg_session in Section 1)
    sess = boc.get_ophys_experiments(cre_lines=[CRE_LINE], targeted_structures=[AREA],
                                     stimuli=[stimulus], imaging_depths=DEPTHS)[0]['id']

    cell_ids, responses, condition, blank, condvals, tuning, blank_mean = compute_tuning(
        boc, sess, stimulus)
    ttest_resp = np.array([is_responsive_ttest(responses[i], condition, blank)[0]
                           for i in range(len(cell_ids))])
    amp_resp = np.nanmax(tuning, axis=0) >= 0.05

    print(f'{stimulus}:  A (t-test) {ttest_resp.sum()}/{len(cell_ids)}   '
          f'B (amplitude) {amp_resp.sum()}/{len(cell_ids)}   '
          f'agree {(ttest_resp == amp_resp).mean():.0%}')

<div style="border-radius: 3px; padding: 10px;">

<b>Beyond yes/no: quantifying <i>how</i> a cell responds.</b> A binary "responsive" label
throws away a lot. Here are a few other ways you could measure responsiveness:
<ul>
<li><b>peak_minus_blank</b> — response magnitude above gray screen</li>
<li><b>snr</b> — magnitude relative to trial-to-trial variability</li>
<li><b>reliability</b> — split-half consistency of the tuning curve</li>
<li><b>response_prob</b> — how <i>often</i> the cell actually responds</li>
<li><b>lifetime_sparseness</b> — how <i>selectively</i> it responds across conditions</li>
<li><b>osi / dsi</b> — orientation / direction selectivity (gratings only)</li>
</ul>
    
Can you think of any other way we may measure responsiveness? Spend some time here....

</div>

In [ ]:
# A few continuous (non-binary) measures of responsiveness.
# Each takes the per-session outputs of compute_tuning and returns one value per cell.

def peak_minus_blank(tuning):
    '''Largest condition response above blank (tuning is already blank-subtracted).'''
    return np.nanmax(tuning, axis=0)


def lifetime_sparseness(tuning):
    '''How selectively a cell responds across conditions
    (0 = responds equally to all, -> 1 = responds to a single condition).'''
    r = np.clip(tuning, 0, None)                 # rectify; tuning is blank-subtracted
    n = r.shape[0]
    num = r.sum(axis=0) ** 2
    den = (r ** 2).sum(axis=0) * n
    with np.errstate(invalid='ignore', divide='ignore'):
        return (1 - num / den) / (1 - 1.0 / n)


def reliability(responses, condition, blank):
    '''Split-half consistency: correlate each cell's tuning curve estimated from
    odd vs even trials. High = the cell responds the same way on repeats.'''
    labels = condition[~blank]
    rr = responses[:, ~blank]                    # (n_cells, n_stim_trials)
    condvals = np.unique(labels)
    half1, half2 = [], []
    for c in condvals:
        idx = np.where(labels == c)[0]
        half1.append(rr[:, idx[0::2]].mean(axis=1))
        half2.append(rr[:, idx[1::2]].mean(axis=1))
    half1 = np.stack(half1, axis=1)              # (n_cells, n_conditions)
    half2 = np.stack(half2, axis=1)
    return np.array([np.corrcoef(half1[i], half2[i])[0, 1]
                     for i in range(half1.shape[0])])

In [ ]:
# Step 1: compute the continuous response measures for the session.
# Step 2: plot their distributions so you can compare variability across cells.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 3c.</b> Do <i>reliable</i> cells tend to be more <i>selective</i>? Select two of your metrics, and plot them as a scatter plot, such as relaibility and lifetime sparseness. What does the distribution of cells look like? Report their correlation.
</div>

In [ ]:
# Step 1: choose two metrics to compare.
# Step 2: make a scatter plot so you can inspect the relationship visually.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


In [ ]:
# Step 1: compute Pearson's r and the p-value for the two metrics.
# Step 2: report the result in a way that can be compared across choices of metrics.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


<div style="border-radius: 3px; padding: 10px;">

So far we've worked through a single example cell. To scale up — and to set up the
cross-stimulus comparison in Section 4 — we need a defined <b>population</b>: the cells that
were recorded for <i>all</i> of our stimuli, matched by <code>cell_specimen_id</code> across
sessions.


<p>The helper <code>get_cells_for_stimuli()</code> builds exactly that. It returns a dataframe
where each row is one such cell, with columns pointing to the exact session that holds that cell's
response to each stimulus (because stimuli live in different sessions, a cell probed with both
<code>drifting_gratings</code> in Session A and <code>natural_scenes</code> in Session B appears in
both under the same <code>cell_specimen_id</code>).



In Exercise 3d we use this table two ways: to
find which sessions to load, and to restrict the analysis to this matched population.

</div>

In [ ]:
def get_cells_for_stimuli(boc, stimuli, cre_lines=None, targeted_structures=None,
                          imaging_depths=None):
    """Cells recorded for *all* of ``stimuli``, matched across sessions.

    Because stimuli live in different sessions of a container, a cell that was
    probed with both ``drifting_gratings`` (session A) and ``natural_scenes``
    (session B) appears in both sessions under the same ``cell_specimen_id``.
    This finds those cells and tells you which session holds each stimulus.

    Parameters
    ----------
    stimuli : list of str
        e.g. ``['drifting_gratings', 'natural_scenes']``.
    cre_lines, targeted_structures, imaging_depths : list, optional
        Standard AllenSDK filters (passed straight through).

    Returns
    -------
    pandas.DataFrame with columns:
        cell_specimen_id, experiment_container_id,
        session_<stimulus>  (one column per requested stimulus)
    """
    filt = {}
    if cre_lines is not None:
        filt['cre_lines'] = cre_lines
    if targeted_structures is not None:
        filt['targeted_structures'] = targeted_structures
    if imaging_depths is not None:
        filt['imaging_depths'] = imaging_depths

    # creates a container dictionary -> {stimulus: session_id}
    container_sessions = {}
    for stim in stimuli:
        for e in boc.get_ophys_experiments(stimuli=[stim], **filt):
            container_sessions.setdefault(e['experiment_container_id'], {})[stim] = e['id']

    rows = []
    for container_id, stim_map in container_sessions.items():
        # keep containers that have a session for EVERY requested stimulus
        if not all(s in stim_map for s in stimuli):
            continue
        # intersect the cell ids across those sessions
        cell_sets = []
        for stim in stimuli:
            ds = boc.get_ophys_experiment_data(stim_map[stim])
            cell_sets.append(set(ds.get_cell_specimen_ids()))
        common = set.intersection(*cell_sets)
        for cid in common:
            row = {'cell_specimen_id': cid,
                   'experiment_container_id': container_id}
            for stim in stimuli:
                row[f'session_{stim}'] = stim_map[stim]
            rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# Step 1: use get_cells_for_stimuli to build the matched population table.
# Step 2: inspect how many experiments are represented in the matched set.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 3d.</b> Calculate criteria A, B and your added metrics for each cell across sessions. How do the results compare across sessions?

</div>

In [ ]:
# Step 1: loop over the sessions that contain the matched cells.
# Step 2: score the same population in each session.
# Step 3: add the summary metrics you want to compare across sessions.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


## 4. Are responses predictive across stimuli?

<div style="border-radius: 3px; padding: 10px;">

Do cells that respond to drifting gratings also respond to natural images? You can answer
this by computing responsiveness directly from ΔF/F for each stimulus,
exactly as in Q3.
<p>The plan:
<ol>
<li>Match cells across the drifting (Session A) and natural-scenes (Session B) sessions
with <code>get_cells_for_stimuli()</code>.</li>
<li>For each stimulus, score every cell as responsive / not, reusing the Q3
criterion. </li>
<li>Merge the two labels per <code>cell_specimen_id</code> and compare.</li>
</ol>
    
We have worked through an example below, but we encourage you to try out different pairs of stimuli and different responsiveness metrics. How do the results compare?

</div>

In [ ]:
# A helper function to measure the Criterion A responsiveness of cells within a session
def session_responsiveness(session_id, stimulus, alpha=0.05):
    _, dff, cids, stim = get_session_dff_and_stim(boc, session_id, stimulus)
    resp = compute_trial_responses(dff, stim)              # (n_cells, n_trials)

    if stimulus == 'drifting_gratings' or stimulus == 'static_gratings':
        cond = stim['orientation'].values                     # blank == NaN orientation
        blank = np.isnan(cond)
    else:  # natural_scenes
        cond = stim['frame'].values                           # blank == frame -1
        blank = (cond == -1)

    # effect size = best condition's mean response minus the blank mean (per cell)
    blank_mean = resp[:, blank].mean(axis=1)
    labels = cond[~blank]
    rr = resp[:, ~blank]
    cond_means = np.stack([rr[:, labels == c].mean(axis=1) for c in np.unique(labels)], axis=1)
    effect = cond_means.max(axis=1) - blank_mean

    # statistical criterion (Q3, criterion A): best condition significantly > blank
    responsive = np.array([is_responsive_ttest(resp[i], cond, blank, alpha)[0]
                           for i in range(len(cids))])

    return pd.DataFrame({'cell_specimen_id': cids.astype(int),
                         'responsive': responsive, 'effect': effect})

<div style="border-radius: 3px; padding: 10px;">

<b>Exercise 4a.</b>
Match cells across the two sessions and score them. Start by using <code>get_cells_for_stimuli</code> for two stimuli you want to compare.

</div>

In [ ]:
# Use the function get_cells_for_stimuli (outlined above) for two stimuli you want to compare.
matched_q4 = get_cells_for_stimuli(
    boc, ['natural_scenes', 'static_gratings'],
    cre_lines=[CRE_LINE], targeted_structures=[AREA], imaging_depths=DEPTHS)

# Note that this will run across experiments, we have set it to run on 3 experiments first.
containers = matched_q4['experiment_container_id'].unique()[:3]
m = matched_q4[matched_q4['experiment_container_id'].isin(containers)].copy()

# How do cells respond (Criterion A) to drifting gratings -- score each relevant session once.
dg = pd.concat([session_responsiveness(s, 'natural_scenes')
                for s in m['session_natural_scenes'].unique()], ignore_index=True)

# How do cells respond (Criterion A) to natural scenes.
ns = pd.concat([session_responsiveness(s, 'static_gratings')
                for s in m['session_static_gratings'].unique()], ignore_index=True)

# Merge the two labels per cell_specimen_id (keeps only the matched cells).
merged = (m.merge(dg.rename(columns={'responsive': 'resp_ns', 'effect': 'eff_ns'}),
                  on='cell_specimen_id')
            .merge(ns.rename(columns={'responsive': 'resp_sg', 'effect': 'eff_sg'}),
                   on='cell_specimen_id'))
print(len(merged), "cells scored for both stimuli")
merged.head()

<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 4a.</b> Can you plot the differences in responsiveness to the two stimuli?
</div>

In [ ]:
# Step 1: plot the responsiveness labels for the two stimuli.
# Step 2: compare how often cells are shared across the two conditions.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


<div style="border-radius: 3px; padding: 10px;">
<b>Exercise 4c.</b> Quantify the association. Can you compare P(responds to natural scenes | responds to drifting) against P(responds to natural scenes | does NOT respond to drifting). Is the difference significant (chi-square test)?
</div>

In [ ]:
# chi-square test on the contingency table + the two conditional probabilities
from scipy.stats import chi2_contingency

table = pd.crosstab(merged['resp_ns'], merged['resp_sg'])
chi2, p, dof, expected = chi2_contingency(table)
print("Contingency table (rows = responds to NS, cols = responds to SG):")
print(table)
print(f"\nchi-square = {chi2:.3f}, dof = {dof}, p = {p:.3g}")

p_sg_given_ns  = merged.loc[merged.resp_ns,  'resp_sg'].mean()
p_sg_given_not = merged.loc[~merged.resp_ns, 'resp_sg'].mean()
print(f"\nP(responds to SG | responds to NS)      = {p_sg_given_ns:.2%}")
print(f"P(responds to SG | does NOT respond to NS) = {p_sg_given_not:.2%}")

<div style="border-radius: 3px; padding: 10px;">

<b>A graded view.</b> Responsive/not is a blunt yes-no. How might you represent this result differently? Can you ask whether a
<i>continuous</i> response property transfers across stimuli: do cells that respond
<i>strongly and reliably</i> to gratings also do so to natural scenes?

<ul>
    <b>A few more questions to continue thinking about:</b>
    <li>Is the correlation different if you compare static gratings and natural scenes?</li>
    <li>How about if you use a different criterion to measure responsiveness?</li>
    <li>What are different ways you might predict a cell's response?</li>
    <li>What if you looked at a different population of cells?</li>
    </ul>
</div>

In [ ]:
# Step 1: compute effect-size measures for the matched cells.
# Step 2: compare the values across stimuli with a scatter plot.
# START YOUR CODE HERE
pass
# END YOUR CODE HERE


## Summary and Continued Project Directions

In this notebook, you explored how a shared population of visual-cortex neurons
responds across multiple stimulus classes and analysis views. You:
- quantified tuning to drifting gratings, static gratings and natural scenes
- compared responsiveness across stimuli (e.g. drifting gratings vs natural scenes)
- examined how response criteria affect which cells are labeled as responsive
- examined correlations between responsiveness of cells across stimuli

### Where to go next in your project

A strong next step is to move from description to prediction.
Here are a few directions you could try:

1. Predict responses from one stimulus type to another
- Build a model that predicts a neuron's response to one stimulus using features from
  another stimulus (e.g: can you predict a cell's response to drifting gratings from
  its response to static gratings?).
- How do different pairs compare? Are some stimuli more predictive than others?
- Compare performance against multi-stimulus models to test whether combining
  stimuli improves prediction.

2. Use multiple response metrics per neuron
- Does adding more response metrics more accurately predict the cell's repsonse?
  (for example: aggregating mean response, response reliability, selectivity index, trial-to-trial variance).
- Test if different combintations of metrics describe the cells more reliably.

3. Analyze predictiveness across different cell-types
- How does the same analysis look on other cell-type populations?
- What are the differences between excitatory and inhibitory neurons? Is the response of one population more consistent/reliable than another population?

4. Analyze population-level structure
- Cluster neurons by combined feature vectors across stimuli.
- Test whether clusters differ in predictability, reliability, or metric profiles.
